In [37]:
import sys
import io

# We patch Jupyter's stdio streams so AnyIO doesn't crash when querying fileno()
try:
    from ipykernel.iostream import OutStream
    OutStream.fileno = lambda self: sys.__stderr__.fileno() if self.name == 'stderr' else sys.__stdout__.fileno()
except ImportError:
    pass

from langchain_google_genai import ChatGoogleGenerativeAI
from typing import List, TypedDict ,Literal,Optional
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from datetime import datetime
load_dotenv()
from pydantic import BaseModel, Field
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_tavily import TavilySearch
import json

In [38]:
import os


SERVERS = {
    "filesystem-server": {
        "transport": "stdio",
        "command": "C:\\Users\\navne\\Desktop\\AMOS\\venv\\Scripts\\python.exe",
        "args": ["C:\\Users\\navne\\Desktop\\AMOS\\mcp\\mcp_filesystem.py"],
        "env": {"MCP_CAPTURE_STDERR": "1", **os.environ}
    }
}

In [39]:

class GoalParser(BaseModel):
    goal: str = Field(
        description="Short machine-friendly goal identifier in snake_case"
    )

    intent: Literal[
        "research",
        "learning",
        "coding",
        "automation",
        "analysis",
        "planning",
        "writing",
        "business_strategy"
    ] = Field(
        description="Primary purpose of the user request"
    )

    priority: Literal["low", "medium", "high"] = Field(
        description="Urgency level inferred from the request"
    )

    domain: List[str] = Field(
        description="Relevant subject domains like AI, finance, healthcare"
    )

    constraints: Optional[List[str]] = Field(
        default=[],
        description="Limitations or conditions mentioned in the request"
    )

    required_agents: List[Literal[
        "research_agent",
        "coding_agent",
        "analysis_agent",
        "automation_agent",
        "browser_agent",
        "planning_agent",
        "data_agent",
        "writing_agent"
    ]] = Field(
        description="Agents needed to complete the task"
    )

    tools_required: List[Literal[
        "web_search",
        "github",
        "filesystem",
        "database",
        "browser",
        "email",
        "api_access"
    ]] = Field(
        description="External tools needed to execute the goal"
    )

    success_criteria: Optional[List[str]] = Field(
        default=[],
        description="Conditions that define task completion"
    )

    estimated_complexity: Literal[
        "simple",
        "moderate",
        "complex"
    ] = Field(
        description="Estimated difficulty of the goal"
    )

    confidence: float = Field(
        ge=0,
        le=1,
        description="Confidence score of the interpretation"
    )

In [40]:
class PlanMetadata(BaseModel):
    plan_id: str = Field(
        description="Unique identifier for the generated plan"
    )

    goal_reference: str = Field(
        description="Goal identifier from the Goal Parser"
    )

    created_at: datetime = Field(
        description="Timestamp when the plan was created"
    )

    estimated_total_steps: int = Field(
        description="Total number of tasks in the plan"
    )

    overall_complexity: Literal[
        "simple",
        "moderate",
        "complex"
    ] = Field(
        description="Estimated complexity of the plan"
    )


class Task(BaseModel):

    task_id: int = Field(
        description="Unique identifier for the task"
    )

    task_description: str = Field(
        description="Clear description of the task"
    )

    assigned_agent: Literal[
        "research_agent",
        "coding_agent",
        "analysis_agent",
        "automation_agent",
        "browser_agent",
        "planning_agent",
        "data_agent",
        "writing_agent"
    ] = Field(
        description="Agent responsible for executing the task"
    )

    tools_required: Optional[List[Literal[
        "web_search",
        "github",
        "filesystem",
        "database",
        "browser",
        "email",
        "api_access"
    ]]] = Field(
        default=[],
        description="Tools required for task execution"
    )

    dependencies: Optional[List[int]] = Field(
        default=[],
        description="Task IDs that must complete before this task"
    )

    priority: Literal[
        "low",
        "medium",
        "high"
    ] = Field(
        description="Task priority level"
    )

    expected_output: str = Field(
        description="Expected result of the task"
    )

    status: Literal[
        "pending",
        "running",
        "completed",
        "failed"
    ] = Field(
        default="pending",
        description="Current status of the task"
    )

    estimated_duration: Optional[int] = Field(
        default=None,
        description="Estimated duration of the task in minutes"
    )

    validation_required: bool = Field(
        default=True,
        description="Whether the critic agent should validate the result"
    )


class ExecutionStrategy(BaseModel):

    execution_mode: Literal[
        "sequential",
        "parallel",
        "hybrid"
    ] = Field(
        description="How tasks should be executed"
    )

    max_retries: int = Field(
        default=2,
        description="Maximum retries allowed for failed tasks"
    )

    validation_required: bool = Field(
        default=True,
        description="Whether outputs must be validated by critic agent"
    )


class PlannerSchema(BaseModel):

    plan_metadata: PlanMetadata = Field(
        description="Metadata describing the generated plan"
    )

    tasks: List[Task] = Field(
        description="List of tasks required to achieve the goal"
    )

    execution_strategy: ExecutionStrategy = Field(
        description="Strategy for executing the tasks"
    )
    

In [41]:
class RouterDecision(BaseModel):

    task_id: int = Field(
        description="ID of the task selected for execution"
    )

    assigned_agent: Literal[
        "research_agent",
        "coding_agent",
        "analysis_agent",
        "automation_agent",
        "browser_agent",
        "planning_agent",
        "data_agent",
        "writing_agent"
    ] = Field(
        description="Agent responsible for executing the task"
    )

    task_description: str = Field(
        description="Description of the task to be executed"
    )

    execution_status: Literal[
        "ready",
        "blocked",
        "completed",
        "failed"
    ] = Field(
        description="Execution state of the task"
    )

    dependencies_satisfied: bool = Field(
        description="Indicates whether all dependencies for this task are completed"
    )

    missing_dependencies: Optional[List[int]] = Field(
        default=[],
        description="List of dependency task IDs that are not yet completed"
    )

    retry_allowed: bool = Field(
        description="Whether the task can be retried if execution fails"
    )

    reason: str = Field(
        description="Explanation of the routing decision"
    )

In [42]:
class Artifact(BaseModel):
    name: str = Field(
        description="Name of the generated artifact"
    )

    type: Literal[
        "file",
        "dataset",
        "report",
        "image",
        "log",
        "model",
        "script"
    ] = Field(
        description="Type of artifact produced by the task"
    )

    location: str = Field(
        description="Path or URL where the artifact is stored"
    )

    description: Optional[str] = Field(
        default=None,
        description="Short explanation of the artifact"
    )


class ExecutionLog(BaseModel):
    timestamp: datetime = Field(
        description="Time when the log entry was created"
    )

    message: str = Field(
        description="Execution message or event"
    )

    level: Literal[
        "info",
        "warning",
        "error"
    ] = Field(
        description="Severity level of the log"
    )


class AgentExecutionResult(BaseModel):

    task_id: int = Field(
        description="ID of the executed task"
    )

    agent: Literal[
        "research_agent",
        "coding_agent",
        "analysis_agent",
        "automation_agent",
        "browser_agent",
        "planning_agent",
        "data_agent",
        "writing_agent"
    ] = Field(
        description="Agent that executed the task"
    )

    execution_status: Literal[
        "completed",
        "failed",
        "partial"
    ] = Field(
        description="Result of the task execution"
    )

    result_summary: str = Field(
        description="Short description of the execution result"
    )

    artifacts: Optional[List[Artifact]] = Field(
        default=[],
        description="Artifacts generated by the task"
    )

    logs: Optional[List[ExecutionLog]] = Field(
        default=[],
        description="Execution logs produced during the task"
    )

    error_message: Optional[str] = Field(
        default=None,
        description="Error message if execution failed"
    )

    started_at: datetime = Field(
        description="Timestamp when execution started"
    )

    completed_at: datetime = Field(
        description="Timestamp when execution finished"
    )

    execution_time_seconds: Optional[int] = Field(
        default=None,
        description="Total execution duration"
    )

    retry_count: int = Field(
        default=0,
        description="Number of retries attempted"
    )

    validation_passed: Optional[bool] = Field(
        default=None,
        description="Whether the critic validated the result"
    )

In [43]:
class CodingPlan(BaseModel):
    files_to_create_or_modify: List[str] = Field(
        description="List of file paths that will be written to or modified."
    )
    libraries_needed: List[str] = Field(
        description="External libraries required (e.g., pip install <lib>)."
    )
    step_by_step_logic: List[str] = Field(
        description="Detailed step-by-step technical plan for what the code needs to do."
    )
    potential_challenges: List[str] = Field(
        description="Potential edge cases or bugs to watch out for."
    )

class State(TypedDict):
    task: str
    goal: Optional[GoalParser]
    plan: Optional[PlannerSchema]
    router_decision: Optional[RouterDecision]
    completed_tasks: List[int]
    current_task_index: int
    queries: List[str]
    coding_plan: Optional[CodingPlan]
    task_results: dict  # To store the results of all tasks

In [44]:
llm=ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
)


In [45]:



GOAL_PARSER="""
You are the **Goal Parser Agent** inside an Autonomous Multi-Agent AI system.

Your responsibility is to convert a user's natural language request into a **structured machine-readable goal object** that other AI agents can execute.

You must carefully analyze the user request and extract the following fields.

Return ONLY valid JSON.

Fields to extract:

goal:
A short machine-friendly goal identifier in snake_case.

intent:
The high-level purpose of the request.
Examples: research, learning, automation, coding, analysis, planning, writing, business_strategy.

domain:
The subject areas involved in the request.
Return a list.

constraints:
Important limitations, requirements, or conditions mentioned by the user.
Return a list. If none exist return an empty list.

priority:
Determine urgency from the user request.
Possible values: low, medium, high.

required_agents:
Which specialized agents will likely be needed to complete this task.
Choose from:

research_agent
coding_agent
analysis_agent
automation_agent
browser_agent
planning_agent
data_agent
writing_agent

If uncertain, infer the most likely agents.

tools_required:
Tools that may be needed. Choose from:

web_search
github
filesystem
database
browser
email
api_access

confidence:
Confidence score between 0 and 1 indicating how certain you are about the interpretation.

Output Format:

{{
"goal": "",
"intent": "",
"domain": [],
"constraints": [],
"priority": "",
"required_agents": [],
"tools_required": [],
"confidence": 0.0
}}

Rules:

1. Return only valid JSON.
2. Do not include explanations.
3. Infer missing information intelligently.
4. Ensure the goal field is short and machine-friendly.
5. Always fill every field.


"""

async def parse_goal(state: State):
    prompt=ChatPromptTemplate.from_messages(
        [
            ("system", GOAL_PARSER),
            ("user", "{question}")
        ]
    )
    parser_llm=llm.with_structured_output(GoalParser)

    parser_chain=prompt | parser_llm
    parsed_goal = await parser_chain.ainvoke({"question": state["task"]})
    return {"goal": parsed_goal}



In [46]:
PLANNER_PROMPT="""
You are the **Planner Agent** inside an Autonomous Multi-Agent AI system.

Your responsibility is to transform a **structured goal object** into a **complete execution plan** that can be executed by multiple specialized AI agents.

The plan must strictly follow the provided schema.

Your plan should break the goal into clear, logical tasks and assign each task to the most appropriate agent.

---

GOAL OBJECT

You will receive a structured goal containing:

* goal
* intent
* priority
* domain
* constraints
* required_agents
* tools_required
* success_criteria
* estimated_complexity
* confidence

Use this information to design the plan.

---

PLANNING RULES

1. Break the goal into **clear atomic tasks**.
2. Each task must have a **specific purpose and expected output**.
3. Assign the most appropriate **agent** to each task.
4. Specify which **tools** are required.
5. Add **dependencies** if tasks must run after others.
6. Tasks should be logically ordered.
7. Avoid redundant tasks.
8. Ensure the final tasks achieve the goal.
9. Use **sequential execution** unless tasks are clearly independent.
10. Make the plan efficient and realistic.

---

AVAILABLE AGENTS

research_agent
coding_agent
analysis_agent
automation_agent
browser_agent
planning_agent
data_agent
writing_agent

---

AVAILABLE TOOLS

web_search
github
filesystem
database
browser
email
api_access

---

EXECUTION STRATEGY

Choose execution mode:

sequential
parallel
hybrid

Use **sequential** if tasks depend on previous steps.

---

OUTPUT FORMAT

Return ONLY valid JSON that matches this structure:

{{
"plan_metadata": {{
"plan_id": "",
"goal_reference": "",
"created_at": "",
"estimated_total_steps": 0,
"overall_complexity": ""
}},

"tasks": [
{{
"task_id": 1,
"task_description": "",
"assigned_agent": "",
"tools_required": [],
"dependencies": [],
"priority": "",
"expected_output": "",
"status": "pending",
"estimated_duration": 0,
"validation_required": true
}}
],

"execution_strategy": {{
"execution_mode": "",
"max_retries": 2,
"validation_required": true
}}
}}

---

IMPORTANT RULES

1. Return **ONLY JSON**.
2. Do not include explanations.
3. Ensure tasks logically achieve the goal.
4. Use valid agents and tools from the provided lists.
5. Ensure dependencies reference valid task IDs.
6. Always fill every required field.
7. Task IDs must start at 1 and increase sequentially.

---

"""


async def create_plan(state: State):
    prompt=ChatPromptTemplate.from_messages(
        [
            ("system", PLANNER_PROMPT),
            ("user", "{goal_object}")
        ]
    )
    planner_llm=llm.with_structured_output(PlannerSchema)

    planner_chain=prompt | planner_llm
    plan = await planner_chain.ainvoke({"goal_object": state["goal"].model_dump_json()})
    return {"plan": plan}

In [47]:
ROUTER_PROMPT="""
You are the **Task Router Agent** inside an Autonomous Multi-Agent AI system.

Your job is to determine **which task should execute next** and **which agent should execute it**.

You will receive:

1. A full execution plan containing tasks.
2. The list of completed tasks.
3. The current task index.

Your responsibility is to analyze the plan and return a routing decision.

---

SYSTEM OBJECTIVE

Select the **next executable task** and route it to the correct agent.

---

ROUTING RULES

1. Always evaluate tasks in order of **task_id**.
2. A task is **ready** only if all its dependencies are completed.
3. If dependencies are missing, mark the task as **blocked**.
4. If the task is completed already, skip it.
5. Route the task to the **assigned_agent** specified in the plan.
6. Only return **one routing decision at a time**.
7. If no task is ready, return the next blocked task with missing dependencies.

---

AVAILABLE AGENTS

research_agent
coding_agent
analysis_agent
automation_agent
browser_agent
planning_agent
data_agent
writing_agent

---

EXECUTION STATUS VALUES

ready
blocked
completed
failed

---

OUTPUT FORMAT

Return ONLY valid JSON matching this structure:

{{
"task_id": 0,
"assigned_agent": "",
"task_description": "",
"execution_status": "",
"dependencies_satisfied": true,
"missing_dependencies": [],
"retry_allowed": true,
"reason": ""
}}

---

OUTPUT RULES

1. Return **only JSON**.
2. Ensure the assigned_agent is valid.
3. If dependencies exist and are not completed, mark execution_status = "blocked".
4. If dependencies are satisfied, mark execution_status = "ready".
5. Explain the decision in the reason field.
6. Do not add explanations outside JSON.

---

PLAN

{plan}

---

COMPLETED TASKS

{completed_tasks}

---

CURRENT TASK INDEX

{current_task_index}
"""

async def route_task(state: State):
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", ROUTER_PROMPT),
            ("user", "Analyze the current state and provide the next routing decision.")
        ]
    )
    
    
    router_llm = llm.with_structured_output(RouterDecision)
    router_chain = prompt | router_llm
    
    decision = await router_chain.ainvoke({
        "plan": state["plan"].model_dump_json() if state.get("plan") else "{}",
        "completed_tasks": json.dumps(state.get("completed_tasks", [])),
        "current_task_index": str(state.get("current_task_index", 0))
    })

    return {"router_decision": decision}


In [48]:

QUERY_GENERATION_PROMPT="""
You are a **Search Query Generator** used inside a research agent.

Your task is to convert a research task into **high quality search engine queries** that will retrieve the most relevant information from the web.

Follow these rules:

1. Generate **3 to 6 different search queries**.
2. Queries should be **specific and information-rich**.
3. Use **keywords that experts would search for**.
4. Avoid conversational phrases.
5. Focus on **technical terms, concepts, and entities**.
6. Each query should target **a slightly different angle of the problem**.
7. Keep each query concise (5–12 words).

Return ONLY a JSON list.

Example output format:

{{
"queries": [
"query 1",
"query 2",
"query 3"
]
}}

Research task:
{task_description}

"""

class QueryList(BaseModel):
    queries: List[str] = Field(description="List of high-quality search engine queries")

async def generate_queries(state: State):
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", QUERY_GENERATION_PROMPT),
            ("user", "Generate search queries for the following research task.")
        ]
    )
    
    query_llm = llm.with_structured_output(QueryList)
    query_chain = prompt | query_llm
    
    queries_obj = await query_chain.ainvoke({
        "task_description": state["router_decision"].task_description if state.get("router_decision") else ""
    })
    
    return {"queries": queries_obj.queries}


In [49]:
CODING_PLANNER_PROMPT = """
You are the **Coding Planner Agent** inside the AMOS framework.

Your task is to properly analyze a coding assignment and generate a structured execution plan before any code is written. 
You must explicitly identify which files need to be modified or created, and the step-by-step logic required.

Task Description:
{task_description}

Previous Context / Research Results:
{task_results}

Create a technical plan detailing the files to write, required libraries, and step-by-step logic.
"""

async def coding_planner(state: State):
    decision = state.get("router_decision")
    if not decision:
        return state
        
    print(f"\n--> [Coding Planner] Planning for Task ID {decision.task_id}: {decision.task_description}")
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", CODING_PLANNER_PROMPT),
        ("user", "Create the coding plan.")
    ])
    
    planner_llm = llm.with_structured_output(CodingPlan)
    planner_chain = prompt | planner_llm
    
    # Pass previous task results for context (e.g. if research was done before coding)
    previous_results = {k: v for k, v in state.get("task_results", {}).items() if str(k) != str(decision.task_id)}
    
    coding_plan = await planner_chain.ainvoke({
        "task_description": decision.task_description,
        "task_results": json.dumps(previous_results, default=str)
    })
    
    return {"coding_plan": coding_plan}

In [50]:
import os

CODE_WRITER_PROMPT = """
You are the **Code Writer Agent** inside the AMOS framework.
You have access to file system tools via MCP. Use them to write or modify files as specified in the plan.

Your task is to write code based strictly on the approved coding plan.

Coding Plan:
{coding_plan}

Write the required code using your tools. Once finished, return a brief text summary of what you did.
"""

async def code_writer(state: State):
    plan = state.get("coding_plan")
    decision = state.get("router_decision")
    if not plan or not decision:
        return state
        
    print(f"\n--> [Code Writer] Writing code for Task ID {decision.task_id}...")
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", CODE_WRITER_PROMPT),
        ("user", "Write the necessary code based on the plan.")
    ])
    
    # Use global mcp_tools populated in main()
    global mcp_tools
    writer_llm = llm.bind_tools(mcp_tools)
    writer_chain = prompt | writer_llm
    
    output = await writer_chain.ainvoke({
        "coding_plan": plan.model_dump_json() if plan else "{}"
    })
    
    # Output content could be a list, string, or empty. Fallback to empty string safely.
    output_content = str(output.content) if output.content else ""
    tool_results_text = []
    
    # Process tool calls if the LLM invoked any
    if hasattr(output, "tool_calls") and output.tool_calls:
        print(f"    Executing {len(output.tool_calls)} tool calls...")
        for tool_call in output.tool_calls:
            print(f"      Calling tool: {tool_call['name']}...")
            # Find the matching tool implementation
            tool_impl = next((t for t in mcp_tools if t.name == tool_call["name"]), None)
            if tool_impl:
                result = await tool_impl.ainvoke(tool_call["args"])
                print(f"      Tool Result: {result}")
                tool_results_text.append(f"Tool {tool_call['name']} Result: {result}")
            else:
                tool_results_text.append(f"Tool {tool_call['name']} not found.")
                
    # Mark task completed
    completed = state.get("completed_tasks", [])
    if decision.task_id not in completed:
        completed.append(decision.task_id)
        
    # Combine standard content with any background tool results
    final_output = output_content + "\n" + "\n".join(tool_results_text)
    if not final_output.strip():
        final_output = "No text content generated."
        
    task_results = state.get("task_results", {})
    task_results[str(decision.task_id)] = final_output
        
    return {
        "completed_tasks": completed,
        "current_task_index": state.get("current_task_index", 0) + 1,
        "task_results": task_results
    }

In [51]:
async def _mock_agent_execution(agent_name: str, state: State):
    decision = state.get("router_decision")
    if not decision:
        return state
        
    print(f"\n--> [{agent_name}] Executing Task ID {decision.task_id}: {decision.task_description}")
    
    # Retrieve results from previous tasks if they exist
    previous_results = state.get("task_results", {})
    
    # Mark task as completed
    completed = state.get("completed_tasks", [])
    if decision.task_id not in completed:
        completed.append(decision.task_id)
        
    # Store dummy results for this agent using the task_id
    task_results = state.get("task_results", {})
    task_results[str(decision.task_id)] = f"Results generated by {agent_name} for task {decision.task_id}"
        
    return {
        "completed_tasks": completed,
        "current_task_index": state.get("current_task_index", 0) + 1,
        "task_results": task_results
    }

async def tavily_search(query: str, max_results: int = 10) -> List[dict]:

    tool = TavilySearch(max_results=max_results)

    results = await tool.ainvoke({"query": query})

    normalized: List[dict] = []

    for r in results.get("results", []):
        normalized.append({
            "title": r.get("title") or "",
            "url": r.get("url") or "",
            "published_at": r.get("published_at") or "",
            "snippet": r.get("snippet") or r.get("content") or "",
            "source": r.get("source") or "",
        })

    return normalized



async def research_agent(state: State): 
    decision = state.get("router_decision")
    if not decision:
        return state
        
    print(f"\n--> [Research Agent] Executing Task ID {decision.task_id}: {decision.task_description}")
    
    # Retrieve results from previous tasks if they exist
    previous_results = state.get("task_results", {})
    
    queries = state.get("queries", [])
    all_results = []
    for query in queries:
        print(f"    Searching for query: {query}")
        results = await tavily_search(query)
        all_results.extend(results)
        
    print(f"    Retrieved {len(all_results)} total results from Tavily Search.")
    
    completed = state.get("completed_tasks", [])
    if decision.task_id not in completed:
        completed.append(decision.task_id)
        
    task_results = state.get("task_results", {})
    task_results[str(decision.task_id)] = all_results
        
    return {
        "completed_tasks": completed,
        "current_task_index": state.get("current_task_index", 0) + 1,
        "task_results": task_results
    }

async def writing_agent(state: State): return await _mock_agent_execution("Writing Agent", state)

async def analysis_agent(state: State): return await _mock_agent_execution("Analysis Agent", state)
async def automation_agent(state: State): return await _mock_agent_execution("Automation Agent", state)
async def browser_agent(state: State): return await _mock_agent_execution("Browser Agent", state)
async def planning_agent(state: State): return await _mock_agent_execution("Planning Agent", state)
async def data_agent(state: State): return await _mock_agent_execution("Data Agent", state)

def route_to_agent(state: State):
    """Conditional edge from router to the chosen agent or END."""
    decision = state.get("router_decision")
    plan = state.get("plan")
    
    # If there's no decision, or all tasks are done, go to END
    if not decision or decision.execution_status == "completed":
        return END
        
    completed_tasks = state.get("completed_tasks", [])
    if plan and len(completed_tasks) >= len(plan.tasks):
        return END

    # Route dynamically based on the assigned agent name
    return decision.assigned_agent

In [52]:
async def main():
    global mcp_tools
    mcp_client = MultiServerMCPClient(SERVERS)
    mcp_tools = await mcp_client.get_tools()
    llm_with_tools = llm.bind_tools(mcp_tools, tool_choice="auto")

    workflow = StateGraph(State)

    workflow.add_node("goal_parser", parse_goal)
    workflow.add_node("planner", create_plan)
    workflow.add_node("router", route_task)

    # Add all the specialized specialized agents
    workflow.add_node("research_agent", research_agent)
    workflow.add_node("coding_planner", coding_planner)
    workflow.add_node("code_writer", code_writer)
    workflow.add_node("analysis_agent", analysis_agent)
    workflow.add_node("automation_agent", automation_agent)
    workflow.add_node("browser_agent", browser_agent)
    workflow.add_node("planning_agent", planning_agent)
    workflow.add_node("data_agent", data_agent)
    workflow.add_node("writing_agent", writing_agent)
    workflow.add_node("generate_queries", generate_queries)

    # Set the entry point and the static edges
    workflow.add_edge(START, "goal_parser")
    workflow.add_edge("goal_parser", "planner")
    workflow.add_edge("planner", "router")

    workflow.add_conditional_edges("router", route_to_agent,{
        "research_agent": "generate_queries",
        "coding_agent": "coding_planner",
        "analysis_agent": "analysis_agent",
        "automation_agent": "automation_agent",
        "browser_agent": "browser_agent",
        "planning_agent": "planning_agent",
        "data_agent": "data_agent",
        "writing_agent": "writing_agent",
        END: END
    })
    workflow.add_edge("generate_queries", "research_agent")

    # Sequence for coding agent execution
    workflow.add_edge("coding_planner", "code_writer")

    agents=[
        "research_agent",
        "code_writer",
        "analysis_agent",
        "automation_agent",
        "browser_agent",
        "planning_agent",
        "data_agent",
        "writing_agent"
    ]

    for agent in agents:
        workflow.add_edge(agent, "router")


    app=workflow.compile()
    print(app)
    initial_state = {
    "task": "Research the best open-source vector databases for AI applications and implement a simple Python example using one of them.",
    "completed_tasks": [],
    "current_task_index": 0,
    "task_results": {}
    }
    result = await app.ainvoke(initial_state)

    printable_result = {
        "task": result["task"],
        "completed_tasks": result.get("completed_tasks"),
        "current_task_index": result.get("current_task_index"),
        "task_results": result.get("task_results"),
        "goal": result["goal"].model_dump() if result.get("goal") else None,
        "plan": result["plan"].model_dump() if result.get("plan") else None,
        "router_decision": result["router_decision"].model_dump() if result.get("router_decision") else None,
    }

    print(json.dumps(printable_result, indent=4, default=str))

In [53]:
# Instead of asyncio.run(), directly 'await' the main function since Jupyter already runs in an event loop
await main()


--> [Research Agent] Executing Task ID 1: Research open-source Python-based vector database options suitable for a simple implementation.
    Searching for query: best lightweight open source python vector databases
    Searching for query: compare chroma vs faiss vs qdrant for python projects
    Searching for query: easy to implement local python vector stores for embeddings
    Searching for query: python vector database libraries for semantic search applications
    Searching for query: top rated open source vector database for python developers
    Retrieved 47 total results from Tavily Search.

--> [Coding Planner] Planning for Task ID 2: Select the best candidate and write a basic Python implementation script to demonstrate document insertion and search.


Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring



--> [Code Writer] Writing code for Task ID 2...
    Executing 1 tool calls...
      Calling tool: write_file...
      Tool Result: [{'type': 'text', 'text': 'File written successfully: C:\\Users\\navne\\Desktop\\requirements.txt', 'id': 'lc_68285e8d-dfe3-4b87-9ea9-45c8ac9a7341'}]

--> [Coding Planner] Planning for Task ID 3: Verify the implementation by running local tests and documenting usage instructions.


Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring



--> [Code Writer] Writing code for Task ID 3...
    Executing 1 tool calls...
      Calling tool: write_file...
      Tool Result: [{'type': 'text', 'text': 'File written successfully: C:\\Users\\navne\\Desktop\\vector_db_demo.py', 'id': 'lc_d5e1f87f-fccd-4c8a-885f-aa14542194cd'}]
{
    "task": "Research the best open-source vector databases for AI applications and implement a simple Python example using one of them.",
    "completed_tasks": [
        1,
        2,
        3
    ],
    "current_task_index": 3,
    "task_results": {
        "1": [
            {
                "title": "Best Open Source Vector Databases 2026 & Comparison - Redis",
                "url": "https://redis.io/blog/best-open-source-vector-databases-comparison/",
                "published_at": "",
                "snippet": "Open source vector databases come in two flavors: specialized tools that handle vectors and nothing else, or unified platforms that combine vector search with operational data and cachin